### **IMPORTS**

In [ ]:
!pip install tab-transformer-pytorch

In [ ]:
import os
import random
import time
import inspect
import numpy as np
import pandas as pd

import glob
import zipfile

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import repeat
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.ensemble import RandomForestClassifier

# Import FTTransformer - we are using "Revisiting Deep Learning Models for Tabular Data" article code,
# from this github link: https://github.com/lucidrains/tab-transformer-pytorch/tree/main?tab=readme-ov-file
from tab_transformer_pytorch import FTTransformer

In [ ]:
# Show FTTransformer "inside"
import inspect
print(inspect.getsource(FTTransformer))

class FTTransformer(Module):
    def __init__(
        self,
        *,
        categories,
        num_continuous,
        dim,
        depth,
        heads,
        dim_head = 16,
        dim_out = 1,
        num_special_tokens = 2,
        attn_dropout = 0.,
        ff_dropout = 0.,
        num_residual_streams = 4
    ):
        super().__init__()
        assert all(map(lambda n: n > 0, categories)), 'number of each category must be positive'
        assert len(categories) + num_continuous > 0, 'input shape must not be null'

        # categories related calculations

        self.num_categories = len(categories)
        self.num_unique_categories = sum(categories)

        # create category embeddings table

        self.num_special_tokens = num_special_tokens
        total_tokens = self.num_unique_categories + num_special_tokens

        # embed

        categories_with_special = tuple(c + num_special_tokens for c in categories)

        self.embedding = Embed(dim, num_discrete =

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

CUDA available: True
CUDA device: Tesla T4


# **Functions**

#### Train / Validation / Test split


In [ ]:
def split_train_val_test(  df, target_col, test_size = 0.1,  val_size= 0.1, random_state = 42, stratify = True):

    """
    - Splits a dataframe into train, validation and test - (by deadult: 80/10/10 and stratifies by the target column)
    """

    if test_size + val_size >= 1.0:
        raise ValueError("test_size + val_size must be < 1.0")

    stratify_col = df[target_col] if stratify else None

    # First split: TRAIN+VAL vs TEST
    train_val_df, test_df = train_test_split( df, test_size=test_size, random_state=random_state, stratify=stratify_col )

    # Adjust stratification for second split
    stratify_col_2 = train_val_df[target_col] if stratify else None

    # Second split: TRAIN vs VAL
    val_fraction_of_train_val = val_size / (1.0 - test_size)

    train_df, val_df = train_test_split( train_val_df, test_size=val_fraction_of_train_val, random_state=random_state, stratify=stratify_col_2 )

    return train_df, val_df, test_df


#### Remove duplicates and report

In [ ]:
# Deduplicate a given df and report duplicates percentage
def deduplicate_dataframe(df):

    n_total = len(df)
    n_duplicates = df.duplicated().sum()
    pct_duplicates = 100.0 * n_duplicates / n_total if n_total > 0 else 0.0
    print(f"Original shape: {df.shape}")
    print(f"Duplicate rows: {n_duplicates}")
    print(f"Duplicate percentage: {pct_duplicates:.2f}%")

    df_unique = df.drop_duplicates().reset_index(drop=True)
    print(f"After deduplication: {df_unique.shape}")

    return df_unique

#### Handling missing values


In [ ]:
# standardize missing values bc in each dataset they may look different (e.g., turn '?' into NaN).

def standardize_missing_values( df , extra_missing_tokens = ("?", "NA", "N/A", "null", "None", "")):
    standard_df = df.copy()
    # Only apply token replacement on object/string-like columns
    obj_cols = standard_df.select_dtypes(include=["object", "string"]).columns
    if len(obj_cols) > 0:
        standard_df[obj_cols] = standard_df[obj_cols].replace(list(extra_missing_tokens), np.nan)

    return standard_df

In [ ]:
# Count missing values (total + per-column)

def missing_report(df, name, top_k = 10):
    per_col = df.isna().sum().sort_values(ascending=False)
    total_missing = int(per_col.sum())
    n_rows, n_cols = df.shape
    cols_with_missing = int((per_col > 0).sum())

    print(f"[{name}] shape = {df.shape}")
    print(f"[{name}] total missing cells = {total_missing}")
    print(f"[{name}] columns with missing = {cols_with_missing}/{n_cols}")

    if cols_with_missing > 0:
        print(f"[{name}] top {min(top_k, cols_with_missing)} columns by missing:")
        display(per_col.head(top_k))
    else:
        print(f"[{name}] no missing values found.")

    return {
        "shape": (n_rows, n_cols),
        "total_missing": total_missing,
        "cols_with_missing": cols_with_missing,
        "per_col_missing": per_col,
    }

In [ ]:
# Identify numeric vs categorical variables (needed for the missing values imputation, as each type of variable is handled differently)

def infer_num_cat_columns(df,  target_col, explicit_cat_cols = None, explicit_num_cols = None ) -> tuple[list[str], list[str]]:
    """
    Returns (numerical_coloumns, categorical_columnss), without the target_col.
    - If explicit lists are provided, we use them.
    - Otherwise, we define:
        numeric: pandas number dtypes
        categorical: object/string/category + boolean
    """

    feature_df = df.drop(columns=[target_col]) # ignoring the target
    all_feature_cols = feature_df.columns.tolist()

    if explicit_num_cols is not None:
        num_cols = list(explicit_num_cols)
    else:
        num_cols = []

    if explicit_cat_cols is not None:
        cat_cols = list(explicit_cat_cols)
    else:
        cat_cols = []

    if explicit_num_cols is not None or explicit_cat_cols is not None:
        num_cols = [] if explicit_num_cols is None else list(explicit_num_cols)
        cat_cols = [] if explicit_cat_cols is None else list(explicit_cat_cols)

        # Safety: remove target if accidentally included
        num_cols = [c for c in num_cols if c != target_col]
        cat_cols = [c for c in cat_cols if c != target_col]

        # keep only existing columns
        num_cols = [c for c in num_cols if c in feature_df.columns]
        cat_cols = [c for c in cat_cols if c in feature_df.columns]

        return num_cols, cat_cols

    # if we didnt explictly provide the lists, identify them manually:
    num_cols = list(feature_df.select_dtypes(include=["number"]).columns)
    cat_cols = list(feature_df.select_dtypes(include=["object", "string", "category", "bool"]).columns)

    return num_cols, cat_cols

In [ ]:
"""
FULL PIPLINE: for each of the 3 sets, we: (1) turn all missing values to Nan.
(2) identify the numric/categorical variables.
(3) fill missing values: for numrical- using the median, for categorial- using the most common.
"""

def impute_splits_with_verification(train_df, val_df, test_df, target_col, explicit_cat_cols=None, explicit_num_cols=None):
    # 1. Standardize
    train_std = standardize_missing_values(train_df)
    val_std = standardize_missing_values(val_df) if val_df is not None else None
    test_std = standardize_missing_values(test_df) if test_df is not None else None

    """
    # Before reports
    print("\n===== Missing values BEFORE imputation =====")
    missing_report(train_std, name="TRAIN(before)")
    if val_std is not None:
        missing_report(val_std, name="VAL(before)")
    if test_std is not None:
        missing_report(test_std, name="TEST(before)")
    """

    # 2. Infer Cols
    num_cols, cat_cols = infer_num_cat_columns(train_std, target_col, explicit_cat_cols, explicit_num_cols)

    # 3. Fit Imputers (Train Only)
    num_imp = SimpleImputer(strategy="median")
    cat_imp = SimpleImputer(strategy="most_frequent")
    if num_cols: num_imp.fit(train_std[num_cols])
    if cat_cols: cat_imp.fit(train_std[cat_cols])

    # 4. Apply
    def apply(df):
        out = df.copy()
        if num_cols: out[num_cols] = num_imp.transform(out[num_cols])
        if cat_cols: out[cat_cols] = cat_imp.transform(out[cat_cols])
        return out

    # AFTER reports
    """
    print("\n===== Missing values AFTER the imputation =====")
    missing_report(train_std, name="TRAIN(after)")
    if val_std is not None:
        missing_report(val_std, name="VAL(after)")
    if test_std is not None:
        missing_report(test_std, name="TEST(after)")
    """

    return {
        "train": apply(train_std), "val": apply(val_std), "test": apply(test_std),
        "num_cols": num_cols, "cat_cols": cat_cols,
        "cat_encoder": {"num_cols": num_cols, "cat_cols": cat_cols} # Placeholder for bundle
    }

#### Numeric Transformers - Z score and Ranking

In [ ]:
# z score norm:

def fit_zscore_scaler(X_train_num, eps = 1e-6) -> dict:
    X = np.asarray(X_train_num, dtype=np.float64)
    mean = X.mean(axis=0)
    std  = X.std(axis=0, ddof=0)
    std = np.maximum(std, eps)
    return {"mean": mean, "std": std}

# Apply z-score scaler (TRAIN-fitted) to any split:
def transform_zscore(X_num, scaler: dict) -> np.ndarray:
    X = np.asarray(X_num, dtype=np.float64)
    out = (X - scaler["mean"]) / scaler["std"] #using the scaler we fitted on the train
    return out.astype(np.float32)


In [ ]:
# Ranking tranformation:

def fit_rank_transformer(X_train_num):
    X = np.asarray(X_train_num, dtype=np.float64)
    sorted_cols = [np.sort(X[:, j]) for j in range(X.shape[1])]
    return {"sorted_cols": sorted_cols, "n": X.shape[0]}

# apply:
def transform_to_rank(X_num, rank_ref: dict) -> np.ndarray:
    X = np.asarray(X_num, dtype=np.float64)
    n = rank_ref["n"]
    out = np.empty_like(X, dtype=np.float32)

    for j, s in enumerate(rank_ref["sorted_cols"]):
        k = np.searchsorted(s, X[:, j], side="right").astype(np.float64)
        p = (k - 0.5) / n
        p = np.clip(p, 1e-6, 1 - 1e-6)
        out[:, j] = p.astype(np.float32)

    return out


#### Numeric Transformers - PLE
* including the embedding and model builder

In [ ]:
# fit quantile edges by the training data

@torch.no_grad()
def fit_ple_edges_quantiles(X_train, n_bins = 20) -> torch.Tensor:
    """
    Computes quantile edges for PLE (picewise linear encoding).
    X_train: (N, D) array or tensor (N= num of rows, D= num of numerical variables)
    Returns: (D, T+1) tensor of edges (T= num of bins)
    """
    X = torch.as_tensor(X_train, dtype=torch.float32)

    # Calculate quantiles:
    qs = torch.linspace(start = 0, end = 1, steps= (n_bins + 1), device=X.device)
    edges = torch.quantile(X, qs, dim=0).T  # (D, n_bins+1)

    # Add a small number to every edge. This guarantees edge[i+1] > edge[i] even if data is all zeros.
    # (We use 1e-4 which is small enough not to distort data).
    epsilon_ramp = torch.linspace(0, 1e-4, n_bins + 1, device=X.device) # = (0.00001, 0.00002, ... 0.0001), we will add them to the edges
    quantiles = edges + epsilon_ramp.unsqueeze(0)

    return quantiles

In [ ]:
"""
explanation about the letters:
B = size of each batch
D = number of numerical variables
T = number of bins we create
dim = the dimension of the embedding
"""

class PLENumericTokenEmbedder(nn.Module):
    def __init__(self, edges: torch.Tensor, dim: int):
        super().__init__()
        D, n_edges = edges.shape # num of numerical variables, num of edges.
        self.T = n_edges - 1    # bc if we have 21 edges we'll have 20 bins.
        self.dim = dim
        self.register_buffer("edges", edges)

        # creating the linear layer, Shape: (D, t, d) means we have matrix for each of the D features, with size t*dim
        self.bin_embed = nn.Parameter(torch.randn(D, self.T, dim) * 0.02)
        self.bias = nn.Parameter(torch.zeros(D, dim))

    def forward(self, x_numer: torch.Tensor) -> torch.Tensor:
        # x_numer: (b, D)
        left  = self.edges[:, :-1]      # (D, t)
        right = self.edges[:, 1:]       # (D, t)

        width = (right - left) + 1e-9   # this add epsilon to width so we won't divide in zero
        x = x_numer.unsqueeze(-1)       # (b, D, 1)

        # ENCODING: the first line defines how much the value "fills" each bin, and the second makes sure that it's between 0-1.
        # for example, for bin [10,20] and value 30, we'll get e= (30-10)/10= 2 --> turn to 1.
        e = (x - left.unsqueeze(0)) / width.unsqueeze(0) # e.g, this results in 2
        e = e.clamp(0.0, 1.0) # this results in 1

        # Weighted Sum - taking the encodded values and put them through a linear layer.
        tokens = torch.einsum("bft,ftd->bfd", e, self.bin_embed) + self.bias.unsqueeze(0) # here f is D (num of features), and d=dim.
        return tokens  # (B, D, dim)

In [ ]:
# Model Wrapper

class FTTransformerWithPLE(nn.Module):
    def __init__(self, base_model: nn.Module, ple_numeric: nn.Module):
        super().__init__()
        self.base = base_model
        self.ple_numeric = ple_numeric

        # Extract components from the lucidrains FTTransformer
        self.embedding = base_model.embedding
        self.transformer = base_model.transformer
        self.to_logits = base_model.to_logits
        self.cls_token = base_model.cls_token
        self.num_special_tokens = base_model.num_special_tokens

    def forward(self, x_categ: torch.Tensor, x_numer: torch.Tensor):
        # 1. Embed Categoricals using the Base Model
        x_categ = x_categ + self.num_special_tokens

        # Lucidrains implementation trick to use only categorical part - We pass None for numeric to bypass the base numeric embedder
        x_cat_tokens = self.embedding(
            (x_categ, None),
            sum_discrete_sets=False,
            sum_continuous=False,
            concat_discrete_continuous=False
        )

        # Handle cases where the wrapper returns a struct or plain tensor
        """
        if not torch.is_tensor(x_cat_tokens):
            x_cat_tokens = x_cat_tokens.discrete
        """

        if x_cat_tokens.ndim == 2:
            x_cat_tokens = x_cat_tokens.unsqueeze(1)

        # 2. Embed Numerics using PLE
        x_num_tokens = self.ple_numeric(x_numer) # (B, D, dim)

        # Concatenate (B, Cat+Num, dim)
        x = torch.cat([x_cat_tokens, x_num_tokens], dim=1)

        # FROM HERE ON- eveyrthing is the same as in the original FTtransformer code
        # Append CLS token
        b = x.shape[0]
        cls_tokens = repeat(self.cls_token, '1 1 d -> b 1 d', b=b)
        x = torch.cat((cls_tokens, x), dim=1)

        # attend
        x = self.transformer(x)    # Transformer Pass

        # get class token
        x = x[:, 0]   # Logits (from CLS token at index 0)
        logits = self.to_logits(x)

        return logits

#### Categorical encoders

In [ ]:
def fit_categorical_encoders(train_df, cat_cols) -> dict:
    """
    Fits per-column category -> integer ID mapping using TRAIN only.
    IDs start at 0 internally, but we return cardinalities + 1 to account for the reserved 'Unknown' token (index 0) used during transform.
    """

    maps = {}
    cardinalities = []
    for c in cat_cols:
        cats = pd.Series(train_df[c].astype(str).unique()).tolist()
        col_map = {v: i for i, v in enumerate(cats)}
        maps[c] = col_map

        # If we have 5 categories, indices will be 1,2,3,4,5 (after shift).
        # Index 0 is reserved for unknown. Total size needed = 6.
        cardinalities.append(len(col_map) + 1)
    return {"cat_cols": list(cat_cols), "maps": maps, "cardinalities": cardinalities}


In [ ]:
# Transform categoricals
def transform_categoricals(df, enc: dict) -> np.ndarray:
    """
    Maps string categories to int IDs.
    - Unknown categories get index 0.
    - Valid categories get index 1, 2, 3...
    """
    cat_cols = enc["cat_cols"] # a list of the categoricals from fit_categorical_encoders
    out = np.empty((len(df), len(cat_cols)), dtype=np.int64)

    for j, c in enumerate(cat_cols):
        col_map = enc["maps"][c]
        vals = df[c].astype(str).values
        out[:, j] = np.array([col_map.get(v, -1) + 1 for v in vals], dtype=np.int64) # here we do the shift

    return out

#### Data Loaders

In [ ]:
# Dataset wrapper to work with categorial + numeric + label from dataframe to tensors to batches

# Dataset wrapper (cat IDs + numeric floats + label)
class TabDataset(Dataset):
    def __init__(self, X_cat, X_num, y):
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)       # For embeddings
        self.X_num = torch.tensor(X_num, dtype=torch.float32)    # For numeric inputs
        self.y = torch.tensor(np.asarray(y), dtype=torch.float32)  # For BCEWithLogitsLoss

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X_cat[i], self.X_num[i], self.y[i]

In [ ]:
# Creates DataLoaders from numpy arrays.  Returns a tuple of (train_loader, val_loader, test_loader)

def build_loaders(Xtr_cat, Xtr_num, ytr, Xva_cat, Xva_num, yva, Xte_cat, Xte_num, yte,
                  batch_size_train=256, batch_size_eval=512, num_workers=0):

    train_ds = TabDataset(Xtr_cat, Xtr_num, ytr)
    val_ds = TabDataset(Xva_cat, Xva_num, yva)
    test_ds = TabDataset(Xte_cat, Xte_num, yte)

    train_loader = DataLoader( train_ds,
                               batch_size=batch_size_train,
                               shuffle=True,
                               num_workers=num_workers )

    val_loader = DataLoader( val_ds,
                             batch_size=batch_size_eval,
                             shuffle=False,
                             num_workers=num_workers )

    test_loader = DataLoader( test_ds,
                              batch_size=batch_size_eval,
                              shuffle=False,
                              num_workers=num_workers )

    return train_loader, val_loader, test_loader


In [ ]:
# Build train/val/test DataLoaders from split DataFrames:

def build_tabular_loaders( train_df, val_df, test_df, target_col, cat_cols,
                          batch_size_train = 256, batch_size_eval = 512, num_workers = 0):

    """
    - Defines numeric columns as: all columns except target_col & cat_cols
    - Extracts (X_cat, X_num, y) for train/val/test
    - Returns DataLoaders + num_cols, and also cardinality of the categorials and number of numrerics.
    """

    # Identify numeric columns (everything else except target + categorical)
    num_cols = [c for c in train_df.columns if c not in cat_cols + [target_col]]

    # Extract arrays
    Xtr_cat = train_df[cat_cols].to_numpy()
    Xva_cat = val_df[cat_cols].to_numpy()
    Xte_cat = test_df[cat_cols].to_numpy()

    Xtr_num = train_df[num_cols].to_numpy(dtype=np.float32)
    Xva_num = val_df[num_cols].to_numpy(dtype=np.float32)
    Xte_num = test_df[num_cols].to_numpy(dtype=np.float32)

    ytr = train_df[target_col].to_numpy()
    yva = val_df[target_col].to_numpy()
    yte = test_df[target_col].to_numpy()

    # Categorical cardinalities (fit on TRAIN only)
    cat_cardinalities = [int(train_df[c].nunique()) for c in cat_cols]

    # Build datasets
    train_ds = TabDataset(Xtr_cat, Xtr_num, ytr)
    val_ds   = TabDataset(Xva_cat, Xva_num, yva)
    test_ds  = TabDataset(Xte_cat, Xte_num, yte)

    # Build loaders
    train_loader = DataLoader(train_ds, batch_size=batch_size_train, shuffle=True,  num_workers=num_workers)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size_eval,  shuffle=False, num_workers=num_workers)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size_eval,  shuffle=False, num_workers=num_workers)

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "cat_cardinalities": cat_cardinalities,
        "num_continuous": len(num_cols),
    }

#### Helper functions for models

In [ ]:
# Setting all the seeds to be the same
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
# Turn off/on the internal numeric normalization
def set_internal_continuous_norm(model, enabled: bool):
    if hasattr(model, "embedding") and hasattr(model.embedding, "can_norm_continuous"):
        model.embedding.can_norm_continuous = bool(enabled)
        return
    raise AttributeError("Could not control internal numeric normalization. "  )


#### Random forest

In [ ]:
def build_rf_matrices( train_df, val_df, test_df, *, target_col, cat_cols ):
    """
    Prepares data for Random Forest:
    - One-hot encodes categoricals (fit on train, align val/test)
    - Keeps numeric features as-is
    - Returns numpy arrays
    """

    # Identify numeric columns
    num_cols = [c for c in train_df.columns if c not in cat_cols + [target_col]]

    # Split into X parts
    Xtr_cat = train_df[cat_cols].astype(str)
    Xva_cat = val_df[cat_cols].astype(str)
    Xte_cat = test_df[cat_cols].astype(str)

    Xtr_num = train_df[num_cols]
    Xva_num = val_df[num_cols]
    Xte_num = test_df[num_cols]

    # One-hot on TRAIN, then align VAL/TEST to TRAIN columns
    Xtr_cat_oh = pd.get_dummies(Xtr_cat, drop_first=False)
    Xva_cat_oh = pd.get_dummies(Xva_cat, drop_first=False)
    Xte_cat_oh = pd.get_dummies(Xte_cat, drop_first=False)

    Xva_cat_oh = Xva_cat_oh.reindex(columns=Xtr_cat_oh.columns, fill_value=0)
    Xte_cat_oh = Xte_cat_oh.reindex(columns=Xtr_cat_oh.columns, fill_value=0)

    # Combine numeric + one-hot categorical
    Xtr = pd.concat([Xtr_num.reset_index(drop=True), Xtr_cat_oh.reset_index(drop=True)], axis=1)
    Xva = pd.concat([Xva_num.reset_index(drop=True), Xva_cat_oh.reset_index(drop=True)], axis=1)
    Xte = pd.concat([Xte_num.reset_index(drop=True), Xte_cat_oh.reset_index(drop=True)], axis=1)

    ytr = train_df[target_col].to_numpy()
    yva = val_df[target_col].to_numpy()
    yte = test_df[target_col].to_numpy()

    return {
        "Xtr": Xtr.to_numpy(),
        "Xva": Xva.to_numpy(),
        "Xte": Xte.to_numpy(),
        "ytr": ytr,
        "yva": yva,
        "yte": yte,
        "num_cols": num_cols,
        "oh_cols": list(Xtr_cat_oh.columns),
        "n_features": Xtr.shape[1],
    }


In [ ]:
# Evaluate ROC-AUC and accuracy **for RF**

def eval_rf_roc_acc(model, X, y, threshold: float = 0.5):
    prob = model.predict_proba(X)[:, 1]
    roc = roc_auc_score(y, prob)
    acc = accuracy_score(y, (prob >= threshold).astype(int))
    return float(roc), float(acc)

In [ ]:
# run (train and test) RF for one seed, with no hyperparameter tuning.
def run_one_seed_rf( seed, Xtr, ytr, Xva, yva, Xte, yte, *,
                               n_estimators=300, max_depth=None, min_samples_leaf = 1,max_features = "sqrt",
                               n_jobs = -1 ):

    clf = RandomForestClassifier( n_estimators=n_estimators,
                                  max_depth=max_depth,
                                  min_samples_leaf=min_samples_leaf,
                                  max_features=max_features,
                                  random_state=seed,
                                  n_jobs=n_jobs,
                                  class_weight="balanced" )

    clf.fit(Xtr, ytr)
    val_roc, _ = eval_rf_roc_acc(clf, Xva, yva)
    test_roc, test_acc = eval_rf_roc_acc(clf, Xte, yte)

    return {
        "seed": seed,
        "val_roc_auc": float(val_roc),
        "test_roc_auc": float(test_roc),
        "test_accuracy": float(test_acc),
        "best_params": {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf,
            "max_features": max_features,
        },
    }

### **FTTransformer: Running one full loop**

In [ ]:
# Predict probabilities on a loader (no gradients)

@torch.no_grad()
def predict_proba(model, loader, device):
    model.eval()
    ys, ps = [], []
    for Xc, Xn, y in loader:
        Xc = Xc.to(device)
        Xn = Xn.to(device)
        logits = model(Xc, Xn).squeeze(1)
        prob = torch.sigmoid(logits).detach().cpu().numpy()
        ys.append(y.numpy())
        ps.append(prob)
    return np.concatenate(ys), np.concatenate(ps)

In [ ]:
# Evaluate ROC-AUC and accuracy **for FT-trandormer**

def eval_roc_acc(model, loader, device, threshold=0.5):
    y_true, y_prob = predict_proba(model, loader, device)
    roc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, (y_prob >= threshold).astype(int))
    return float(roc), float(acc)

In [ ]:
def run_one_seed_ftt(seed, train_loader, val_loader, test_loader, model_builder_fn,
                     epochs, lr=1e-4, weight_decay=1e-4):

    """
    Trains FT-Transformer for one seed.
    Selects best epoch by validation ROC-AUC, reports test metrics.
    model_builder_fn - function that returns a new model instance.
    RETURNS a dict with: val_roc, test_roc_auc, test_accuracy, seed.
    """

    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = model_builder_fn().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

    best_val_roc = 0.0
    best_test_metrics  = {}

    for epoch in range(epochs):
        # 1- train
        model.train()
        for Xc, Xn, y in train_loader:
            Xc, Xn, y = Xc.to(device), Xn.to(device), y.to(device).float()
            optimizer.zero_grad()
            logits = model(Xc, Xn).squeeze(1)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

        # 2- Validation
        val_roc, val_acc = eval_roc_acc(model, val_loader, device)

        # 3- Save the current best (by ROC on the validation set)
        if val_roc > best_val_roc:
            best_val_roc = val_roc
            test_roc, test_acc = eval_roc_acc(model, test_loader, device)
            best_test_metrics  = {
                "val_roc": val_roc,
                "test_roc_auc": test_roc,
                "test_accuracy": test_acc,
                "seed": seed
            }

    return best_test_metrics

In [ ]:
# Summary printer (mean ± std over seeds)

def summarize_results(name, results):
    rocs = np.array([r["test_roc_auc"] for r in results], dtype=float)
    accs = np.array([r["test_accuracy"] for r in results], dtype=float)
    print(f"\n=== {name} Summary (mean ± std over seeds) ===")
    print(f"TEST ROC-AUC: {rocs.mean():.4f} ± {rocs.std(ddof=1):.4f}")
    print(f"TEST ACC:     {accs.mean():.4f} ± {accs.std(ddof=1):.4f}")

## **Helpers for experiment running**

In [ ]:
def run_ftt_method(method_name, dataset_name, cardinalities,
                   Xtr_cat, Xtr_num, ytr,
                   Xva_cat, Xva_num, yva,
                   Xte_cat, Xte_num, yte,
                   model_builder_fn, seeds, epochs):

    """
    Generic runner for any FT-Transformer method (raw, z, ranking, ple).
    (1) build the 3 loaders;
    (2) run s iteration- for each seed we train, evaluate and test the current model.
        * calculates the time needed for each iteration
    (3) report final results.
    """

    print(f"\nRunning {dataset_name} - {method_name}...")
    t0 = time.time()

    # Build loaders
    train_loader, val_loader, test_loader = build_loaders(  Xtr_cat, Xtr_num, ytr,
                                                          Xva_cat, Xva_num, yva,
                                                            Xte_cat, Xte_num, yte )

    # Run across seeds
    results = []
    for s in seeds:
        r = run_one_seed_ftt(s, train_loader, val_loader, test_loader,
                             model_builder_fn, epochs)
        results.append(r)
        print(f"  seed={s} | TEST ROC={r['test_roc_auc']:.4f} | ACC={r['test_accuracy']:.4f}")

    elapsed_min = (time.time() - t0) / 60.0

    # Summary
    summarize_results(f"{dataset_name}_{method_name}", results)
    print(f"Time: {elapsed_min:.2f} min")

    return results, elapsed_min

In [ ]:
def run_dataset_experiment(  dataset_name,
                           Xtr_cat, Xtr_num, ytr,
                             Xva_cat, Xva_num, yva,
                             Xte_cat, Xte_num, yte,
                             cardinalities,
                             *,
                             # RF inputs:
                             train_df, val_df, test_df,  target_col, cat_cols,
                             # FT-Transformer hyperparameters:
                             ftt_dim=32,  ftt_depth=3,  ftt_heads=8,  ftt_attn_dropout=0.2,
                             ftt_ff_dropout=0.2,  ple_n_bins=20,
                             # RF hyperparameters:
                             rf_n_estimators=100,  rf_max_depth=None, rf_min_samples_leaf=1,
                             rf_max_features="sqrt", rf_n_jobs=-1,
                             # Experiment config
                             seeds=(0, 1, 2),  epochs=40 ):
    """
    Runs complete experiment for one dataset:
    1. FT-Transformer with RAW numerics
    2. FT-Transformer with Z-score normalization
    3. FT-Transformer with Rank transformation
    4. FT-Transformer with PLE embeddings
    5. Random Forest baseline
    """
    all_results = {}
    all_times = {}

    # building a standard FT-Transformer for RAW/Z-score/Rank:
    def build_standard_ftt():
        return FTTransformer( categories=tuple(cardinalities),
                             num_continuous=Xtr_num.shape[1],
                              dim=ftt_dim,
                              dim_out=1,
                              depth=ftt_depth,
                              heads=ftt_heads,
                              attn_dropout=ftt_attn_dropout,
                              ff_dropout=ftt_ff_dropout  )


    # buildingFT-Transformer with PLE embeddings for numeric:
    def build_ple_ftt():
        # Fit PLE edges on raw training data
        edges = fit_ple_edges_quantiles(Xtr_num, n_bins=ple_n_bins)

        # Base FT-Transformer with no numeric features (PLE replaces them)
        base = FTTransformer( categories=tuple(cardinalities),
                             num_continuous=0,  # PLE handles the continues
                              dim=ftt_dim,
                              dim_out=1,
                              depth=ftt_depth,
                              heads=ftt_heads,
                              attn_dropout=ftt_attn_dropout,
                              ff_dropout=ftt_ff_dropout,
        )

        # PLE embedder
        ple_embedder = PLENumericTokenEmbedder(edges=edges, dim=ftt_dim)

        return FTTransformerWithPLE(base, ple_embedder)


    # 1) RAW (no normalization)
    res, time_min = run_ftt_method( "RAW", dataset_name, cardinalities,
                                   Xtr_cat, Xtr_num, ytr,
                                    Xva_cat, Xva_num, yva,
                                    Xte_cat, Xte_num, yte,
                                    build_standard_ftt, seeds, epochs )
    all_results["FTT_RAW"] = res
    all_times["FTT_RAW"] = time_min


    # 2) Z-SCORE normalization
    z_scaler = fit_zscore_scaler(Xtr_num)
    Xtr_z = transform_zscore(Xtr_num, z_scaler)
    Xva_z = transform_zscore(Xva_num, z_scaler)
    Xte_z = transform_zscore(Xte_num, z_scaler)

    res, time_min = run_ftt_method( "ZSCORE", dataset_name, cardinalities,
                                   Xtr_cat, Xtr_z, ytr,
                                    Xva_cat, Xva_z, yva,
                                    Xte_cat, Xte_z, yte,
                                    build_standard_ftt, seeds, epochs )
    all_results["FTT_ZSCORE"] = res
    all_times["FTT_ZSCORE"] = time_min


    # 3) RANK transformation
    rank_scaler = fit_rank_transformer(Xtr_num)
    Xtr_rank = transform_to_rank(Xtr_num, rank_scaler)
    Xva_rank = transform_to_rank(Xva_num, rank_scaler)
    Xte_rank = transform_to_rank(Xte_num, rank_scaler)

    res, time_min = run_ftt_method( "RANK", dataset_name, cardinalities,
                                   Xtr_cat, Xtr_rank, ytr,
                                    Xva_cat, Xva_rank, yva,
                                    Xte_cat, Xte_rank, yte,
                                    build_standard_ftt, seeds, epochs )
    all_results["FTT_RANK"] = res
    all_times["FTT_RANK"] = time_min


    # 4) PLE
    # Note: PLE uses RAW numerics (no preprocessing)
    res, time_min = run_ftt_method( "PLE", dataset_name, cardinalities,
                                   Xtr_cat, Xtr_num, ytr,  # Raw numerics
                                    Xva_cat, Xva_num, yva,
                                    Xte_cat, Xte_num, yte,
                                    build_ple_ftt, seeds, epochs )
    all_results["FTT_PLE"] = res
    all_times["FTT_PLE"] = time_min


    # 5) Random Forest baseline
    if train_df is not None:
        print(f"\nRunning {dataset_name} - RF_RAW...")
        t0 = time.time()

        # Prepare RF matrices (one-hot encode categoricals)
        mats = build_rf_matrices( train_df=train_df,
                                 val_df=val_df,
                                  test_df=test_df,
                                  target_col=target_col,
                                  cat_cols=cat_cols )

        print(f"  RF feature count: {mats['n_features']}")

        # Run across seeds
        res_rf = []
        for s in seeds:
            r = run_one_seed_rf( seed=s,
                                Xtr=mats["Xtr"], ytr=mats["ytr"],
                                 Xva=mats["Xva"], yva=mats["yva"],
                                 Xte=mats["Xte"], yte=mats["yte"],
                                 n_estimators=rf_n_estimators,
                                 max_depth=rf_max_depth,
                                 min_samples_leaf=rf_min_samples_leaf,
                                 max_features=rf_max_features,
                                 n_jobs=rf_n_jobs )

            res_rf.append(r)
            print(f"  seed={s} | TEST ROC={r['test_roc_auc']:.4f} | ACC={r['test_accuracy']:.4f}")

        elapsed_min = (time.time() - t0) / 60.0
        all_times["RF_RAW"] = elapsed_min

        summarize_results(f"{dataset_name}_RF_RAW", res_rf)
        print(f"Time: {elapsed_min:.2f} min")

        all_results["RF_RAW"] = res_rf

    # Store timing info
    all_results["_time_minutes"] = all_times

    return all_results



# **FULL EXPERIMENTS**

In [ ]:
!pip install kaggle

os.environ["KAGGLE_USERNAME"] = "YoniSeleznev"
os.environ["KAGGLE_KEY"] = "KGAT_efa45bc834e353496a530f28983f672a"

DATA_ROOT = "data_kaggle"
os.makedirs(DATA_ROOT, exist_ok=True)

# Download + unzip Kaggle "dataset"
def kaggle_dataset(slug: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    !kaggle datasets download -d {slug} -p {out_dir}
    for z in glob.glob(os.path.join(out_dir, "*.zip")):
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(out_dir)

# Download + unzip Kaggle "competition"
def kaggle_competition(comp: str, out_dir: str):
    os.makedirs(out_dir, exist_ok=True)
    !kaggle competitions download -c {comp} -p {out_dir}
    for z in glob.glob(os.path.join(out_dir, "*.zip")):
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall(out_dir)


In [ ]:
# Heart Disease (UCI)

kaggle_dataset("johnsmith88/heart-disease-dataset", f"{DATA_ROOT}/heart")
!ls -lah {DATA_ROOT}/heart | head
heart = pd.read_csv(f"{DATA_ROOT}/heart/heart.csv")
print(heart.shape)
heart.head()

# 1. Load and preprocess data
heart = deduplicate_dataframe(heart)
HEART_TARGET_COL = "target"
HEART_CAT_COLS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

# 2. Split data
heart_train_df, heart_val_df, heart_test_df = split_train_val_test( heart, HEART_TARGET_COL, test_size=0.1, val_size=0.1,
                                                                   random_state=42, stratify=True )

# 3. Impute missing values
HEART_NUM_COLS = [
    c for c in heart_train_df.columns
    if c != HEART_TARGET_COL and c not in HEART_CAT_COLS
]

heart_imputed = impute_splits_with_verification( train_df=heart_train_df,  val_df=heart_val_df, test_df=heart_test_df,
                                                 target_col=HEART_TARGET_COL,
                                                 explicit_cat_cols=HEART_CAT_COLS,
                                                 explicit_num_cols=HEART_NUM_COLS )

ht_tr = heart_imputed["train"]
ht_va = heart_imputed["val"]
ht_te = heart_imputed["test"]

# 4. Encode categoricals
enc_heart = fit_categorical_encoders(ht_tr, HEART_CAT_COLS)
Xtr_cat = transform_categoricals(ht_tr, enc_heart)
Xva_cat = transform_categoricals(ht_va, enc_heart)
Xte_cat = transform_categoricals(ht_te, enc_heart)

# 5. Extract numerics
Xtr_num = ht_tr[HEART_NUM_COLS].values.astype(np.float32)
Xva_num = ht_va[HEART_NUM_COLS].values.astype(np.float32)
Xte_num = ht_te[HEART_NUM_COLS].values.astype(np.float32)

# 6. Extract targets
ytr = ht_tr[HEART_TARGET_COL].values
yva = ht_va[HEART_TARGET_COL].values
yte = ht_te[HEART_TARGET_COL].values

# 7. Run experiment
HEART_RESULTS = run_dataset_experiment(
    dataset_name="Heart",
    Xtr_cat=Xtr_cat, Xtr_num=Xtr_num, ytr=ytr,
    Xva_cat=Xva_cat, Xva_num=Xva_num, yva=yva,
    Xte_cat=Xte_cat, Xte_num=Xte_num, yte=yte,
    cardinalities=enc_heart["cardinalities"],
    train_df=ht_tr,
    val_df=ht_va,
    test_df=ht_te,
    target_col=HEART_TARGET_COL,
    cat_cols=HEART_CAT_COLS,
    seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    epochs= 40
)

# 8. Access results
print("\n=== FINAL RESULTS ===")
for method, results in HEART_RESULTS.items():
    if method != "_time_minutes":
        rocs = [r["test_roc_auc"] for r in results]
        print(f"{method}: {np.mean(rocs):.4f} ± {np.std(rocs, ddof=1):.4f}")


Dataset URL: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset
License(s): unknown
  0% 0.00/6.18k [00:00<?, ?B/s]
100% 6.18k/6.18k [00:00<00:00, 30.7MB/s]
total 56K
drwxr-xr-x 2 root root 4.0K Mar  5 19:28 .
drwxr-xr-x 3 root root 4.0K Mar  5 19:28 ..
-rw-r--r-- 1 root root  38K Mar  5 19:28 heart.csv
-rw-r--r-- 1 root root 6.2K Oct 25  2019 heart-disease-dataset.zip
(1025, 14)
Original shape: (1025, 14)
Duplicate rows: 723
Duplicate percentage: 70.54%
After deduplication: (302, 14)

Running Heart - RAW...
  seed=0 | TEST ROC=0.8487 | ACC=0.8065
  seed=1 | TEST ROC=0.8571 | ACC=0.7742
  seed=2 | TEST ROC=0.8403 | ACC=0.7742
  seed=3 | TEST ROC=0.8613 | ACC=0.7742
  seed=4 | TEST ROC=0.8445 | ACC=0.8065
  seed=5 | TEST ROC=0.8445 | ACC=0.8065
  seed=6 | TEST ROC=0.8571 | ACC=0.8065
  seed=7 | TEST ROC=0.8613 | ACC=0.7742
  seed=8 | TEST ROC=0.8739 | ACC=0.8710
  seed=9 | TEST ROC=0.8487 | ACC=0.8387

=== Heart_RAW Summary (mean ± std over seeds) ===
TEST ROC-AUC: 0.8538

In [ ]:
# CREDIT CARD

kaggle_dataset(
    "uciml/default-of-credit-card-clients-dataset",
    f"{DATA_ROOT}/credit"
)
!ls -lah {DATA_ROOT}/credit | head
credit = pd.read_csv(f"{DATA_ROOT}/credit/UCI_Credit_Card.csv")

# Target + categorical columns
CREDIT_TARGET_COL = "default.payment.next.month"
credit[CREDIT_TARGET_COL] = credit[CREDIT_TARGET_COL].astype(int)

CREDIT_CAT_COLS = [
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"
]

# Drop ID (not a real feature)
credit = credit.drop(columns=["ID"])

# Remove duplicates
credit = deduplicate_dataframe(credit)

# Split
credit_train_df, credit_val_df, credit_test_df = split_train_val_test( credit, CREDIT_TARGET_COL,  test_size=0.1, val_size=0.1,
                                                                      random_state=42, stratify=True )

# Identify numeric columns
CREDIT_NUM_COLS = [
    c for c in credit_train_df.columns
    if c != CREDIT_TARGET_COL and c not in CREDIT_CAT_COLS
]

# Impute
credit_imputed = impute_splits_with_verification( train_df=credit_train_df, val_df=credit_val_df, test_df=credit_test_df,
                                                  target_col=CREDIT_TARGET_COL, explicit_cat_cols=CREDIT_CAT_COLS,
                                                  explicit_num_cols=CREDIT_NUM_COLS)

# Unpack
cr_tr = credit_imputed["train"]
cr_va = credit_imputed["val"]
cr_te = credit_imputed["test"]

# Encode categoricals
enc_c = fit_categorical_encoders(cr_tr, CREDIT_CAT_COLS)
Xtr_cat = transform_categoricals(cr_tr, enc_c)
Xva_cat = transform_categoricals(cr_va, enc_c)
Xte_cat = transform_categoricals(cr_te, enc_c)

# Extract numerics
Xtr_num = cr_tr[CREDIT_NUM_COLS].values.astype(np.float32)
Xva_num = cr_va[CREDIT_NUM_COLS].values.astype(np.float32)
Xte_num = cr_te[CREDIT_NUM_COLS].values.astype(np.float32)

# Extract targets
ytr = cr_tr[CREDIT_TARGET_COL].values
yva = cr_va[CREDIT_TARGET_COL].values
yte = cr_te[CREDIT_TARGET_COL].values

# Run experiment
CARDS_RESULTS = run_dataset_experiment( dataset_name="Credit_card",
                                        Xtr_cat=Xtr_cat, Xtr_num=Xtr_num, ytr=ytr,
                                        Xva_cat=Xva_cat, Xva_num=Xva_num, yva=yva,
                                        Xte_cat=Xte_cat, Xte_num=Xte_num, yte=yte,
                                        cardinalities=enc_c["cardinalities"],
                                        train_df=cr_tr, val_df=cr_va, test_df=cr_te,
                                        target_col=CREDIT_TARGET_COL,
                                        cat_cols=CREDIT_CAT_COLS,
                                        seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
                                        epochs= 40 )

# Access results
print("\n=== FINAL RESULTS ===")
for method, results in CARDS_RESULTS.items():
    if method != "_time_minutes":
        rocs = [r["test_roc_auc"] for r in results]
        print(f"{method}: {np.mean(rocs):.4f} ± {np.std(rocs, ddof=1):.4f}")


Dataset URL: https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
License(s): CC0-1.0
default-of-credit-card-clients-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
total 3.8M
drwxr-xr-x 2 root root  4.0K Feb 25 13:09 .
drwxr-xr-x 3 root root  4.0K Feb 25 13:09 ..
-rw-r--r-- 1 root root 1002K Sep 20  2019 default-of-credit-card-clients-dataset.zip
-rw-r--r-- 1 root root  2.8M Feb 25 15:15 UCI_Credit_Card.csv
Original shape: (30000, 24)
Duplicate rows: 35
Duplicate percentage: 0.12%
After deduplication: (29965, 24)

Running Credit_card - RAW...
  seed=0 | TEST ROC=0.7484 | ACC=0.8168
  seed=1 | TEST ROC=0.7334 | ACC=0.8165
  seed=2 | TEST ROC=0.7523 | ACC=0.8215
  seed=3 | TEST ROC=0.7469 | ACC=0.8168
  seed=4 | TEST ROC=0.7528 | ACC=0.8182
  seed=5 | TEST ROC=0.7513 | ACC=0.8192
  seed=6 | TEST ROC=0.7550 | ACC=0.8178
  seed=7 | TEST ROC=0.7500 | ACC=0.8225
  seed=8 | TEST ROC=0.7581 | ACC=0.8185
  seed=9 | TEST ROC

In [ ]:
# Stroke (Kaggle)

kaggle_dataset("fedesoriano/stroke-prediction-dataset", f"{DATA_ROOT}/stroke")
!ls -lah {DATA_ROOT}/stroke | head

stroke = pd.read_csv(f"{DATA_ROOT}/stroke/healthcare-dataset-stroke-data.csv")
print(stroke.shape)
stroke.head()

# 1. Load and preprocess data
stroke = deduplicate_dataframe(stroke)

STROKE_TARGET_COL = "stroke"

STROKE_CAT_COLS = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

# 2. Split data
stroke_train_df, stroke_val_df, stroke_test_df = split_train_val_test(
    stroke,
    STROKE_TARGET_COL,
    test_size=0.1,
    val_size=0.1,
    random_state=42,
    stratify=True
)

# 3. Impute missing values
STROKE_NUM_COLS = [
    c for c in stroke_train_df.columns
    if c != STROKE_TARGET_COL and c not in STROKE_CAT_COLS
]

stroke_imputed = impute_splits_with_verification(
    train_df=stroke_train_df,
    val_df=stroke_val_df,
    test_df=stroke_test_df,
    target_col=STROKE_TARGET_COL,
    explicit_cat_cols=STROKE_CAT_COLS,
    explicit_num_cols=STROKE_NUM_COLS
)

st_tr = stroke_imputed["train"]
st_va = stroke_imputed["val"]
st_te = stroke_imputed["test"]

# 4. Encode categoricals
enc_stroke = fit_categorical_encoders(st_tr, STROKE_CAT_COLS)

Xtr_cat = transform_categoricals(st_tr, enc_stroke)
Xva_cat = transform_categoricals(st_va, enc_stroke)
Xte_cat = transform_categoricals(st_te, enc_stroke)

# 5. Extract numerics
Xtr_num = st_tr[STROKE_NUM_COLS].values.astype(np.float32)
Xva_num = st_va[STROKE_NUM_COLS].values.astype(np.float32)
Xte_num = st_te[STROKE_NUM_COLS].values.astype(np.float32)

# 6. Extract targets
ytr = st_tr[STROKE_TARGET_COL].values
yva = st_va[STROKE_TARGET_COL].values
yte = st_te[STROKE_TARGET_COL].values

# 7. Run experiment
STROKE_RESULTS = run_dataset_experiment(
    dataset_name="Stroke",
    Xtr_cat=Xtr_cat, Xtr_num=Xtr_num, ytr=ytr,
    Xva_cat=Xva_cat, Xva_num=Xva_num, yva=yva,
    Xte_cat=Xte_cat, Xte_num=Xte_num, yte=yte,
    cardinalities=enc_stroke["cardinalities"],
    train_df=st_tr,
    val_df=st_va,
    test_df=st_te,
    target_col=STROKE_TARGET_COL,
    cat_cols=STROKE_CAT_COLS,
    seeds=[0,1,2,3,4,5,6,7,8,9],
    epochs=40
)

# 8. Access results
print("\n=== FINAL RESULTS ===")

for method, results in STROKE_RESULTS.items():
    if method != "_time_minutes":
        rocs = [r["test_roc_auc"] for r in results]
        print(f"{method}: {np.mean(rocs):.4f} ± {np.std(rocs, ddof=1):.4f}")

Dataset URL: https://www.kaggle.com/datasets/fedesoriano/stroke-prediction-dataset
License(s): copyright-authors
stroke-prediction-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
total 388K
drwxr-xr-x 2 root root 4.0K Mar  5 20:49 .
drwxr-xr-x 5 root root 4.0K Mar  5 21:34 ..
-rw-r--r-- 1 root root 310K Mar  5 21:44 healthcare-dataset-stroke-data.csv
-rw-r--r-- 1 root root  68K Jan 26  2021 stroke-prediction-dataset.zip
(5110, 12)
Original shape: (5110, 12)
Duplicate rows: 0
Duplicate percentage: 0.00%
After deduplication: (5110, 12)

Running Stroke - RAW...
  seed=0 | TEST ROC=0.7233 | ACC=0.9511
  seed=1 | TEST ROC=0.6392 | ACC=0.9511
  seed=2 | TEST ROC=0.6658 | ACC=0.9511
  seed=3 | TEST ROC=0.7226 | ACC=0.9511
  seed=4 | TEST ROC=0.7188 | ACC=0.9511
  seed=5 | TEST ROC=0.6400 | ACC=0.9511
  seed=6 | TEST ROC=0.7030 | ACC=0.9511
  seed=7 | TEST ROC=0.7260 | ACC=0.9511
  seed=8 | TEST ROC=0.6365 | ACC=0.9511
  seed=9 | TEST ROC=0.6217 |

In [ ]:
# Bank Marketing

kaggle_dataset(
    "mohitjanbandhu/bank-marketing-with-socialeconomic-context",
    f"{DATA_ROOT}/bank"
)
!ls -lah {DATA_ROOT}/bank | head

bank = pd.read_csv(f"{DATA_ROOT}/bank/bank-additional-full.csv", sep=";") # the data is ';' separated
print(bank.shape)
bank.head()

# 1. Load and preprocess data
bank = deduplicate_dataframe(bank)

# remove leakage feature
bank = bank.drop(columns=["duration"])

# convert target to 0/1
bank["y"] = (bank["y"] == "yes").astype(int)

BANK_TARGET_COL = "y"

BANK_CAT_COLS = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

# 2. Split data
bank_train_df, bank_val_df, bank_test_df = split_train_val_test(
    bank,
    BANK_TARGET_COL,
    test_size=0.1,
    val_size=0.1,
    random_state=42,
    stratify=True
)

# 3. Impute missing values
BANK_NUM_COLS = [
    c for c in bank_train_df.columns
    if c != BANK_TARGET_COL and c not in BANK_CAT_COLS
]

bank_imputed = impute_splits_with_verification(
    train_df=bank_train_df,
    val_df=bank_val_df,
    test_df=bank_test_df,
    target_col=BANK_TARGET_COL,
    explicit_cat_cols=BANK_CAT_COLS,
    explicit_num_cols=BANK_NUM_COLS
)

bk_tr = bank_imputed["train"]
bk_va = bank_imputed["val"]
bk_te = bank_imputed["test"]

# 4. Encode categoricals
enc_bank = fit_categorical_encoders(bk_tr, BANK_CAT_COLS)

Xtr_cat = transform_categoricals(bk_tr, enc_bank)
Xva_cat = transform_categoricals(bk_va, enc_bank)
Xte_cat = transform_categoricals(bk_te, enc_bank)

# 5. Extract numerics
Xtr_num = bk_tr[BANK_NUM_COLS].values.astype(np.float32)
Xva_num = bk_va[BANK_NUM_COLS].values.astype(np.float32)
Xte_num = bk_te[BANK_NUM_COLS].values.astype(np.float32)

# 6. Extract targets
ytr = bk_tr[BANK_TARGET_COL].values
yva = bk_va[BANK_TARGET_COL].values
yte = bk_te[BANK_TARGET_COL].values

# 7. Run experiment
BANK_RESULTS = run_dataset_experiment(
    dataset_name="BankMarketing",
    Xtr_cat=Xtr_cat, Xtr_num=Xtr_num, ytr=ytr,
    Xva_cat=Xva_cat, Xva_num=Xva_num, yva=yva,
    Xte_cat=Xte_cat, Xte_num=Xte_num, yte=yte,
    cardinalities=enc_bank["cardinalities"],
    train_df=bk_tr,
    val_df=bk_va,
    test_df=bk_te,
    target_col=BANK_TARGET_COL,
    cat_cols=BANK_CAT_COLS,
    seeds=[0,1,2,3,4,5,6,7,8,9],
    epochs=40
)

# 8. Access results
print("\n=== FINAL RESULTS ===")
for method, results in BANK_RESULTS.items():
    if method != "_time_minutes":
        rocs = [r["test_roc_auc"] for r in results]
        print(f"{method}: {np.mean(rocs):.4f} ± {np.std(rocs, ddof=1):.4f}")

Dataset URL: https://www.kaggle.com/datasets/mohitjanbandhu/bank-marketing-with-socialeconomic-context
License(s): apache-2.0
bank-marketing-with-socialeconomic-context.zip: Skipping, found more recently modified local copy (use --force to force download)
total 18M
drwxr-xr-x 2 root root 4.0K Mar  5 21:34 .
drwxr-xr-x 5 root root 4.0K Mar  5 21:34 ..
-rw-r--r-- 1 root root 4.8M Mar  5 22:41 bank-addi-full_portuges.csv
-rw-r--r-- 1 root root 5.6M Mar  5 22:41 bank-additional-full.csv
-rw-r--r-- 1 root root 1.2M Feb  2  2024 bank-marketing-with-socialeconomic-context.zip
-rw-r--r-- 1 root root 5.6M Mar  5 22:41 DummiesFile.csv
(41188, 21)
Original shape: (41188, 21)
Duplicate rows: 12
Duplicate percentage: 0.03%
After deduplication: (41176, 21)

Running BankMarketing - RAW...
  seed=0 | TEST ROC=0.8013 | ACC=0.8987
  seed=1 | TEST ROC=0.8010 | ACC=0.8978
  seed=2 | TEST ROC=0.8027 | ACC=0.8968
  seed=3 | TEST ROC=0.8041 | ACC=0.8963
  seed=4 | TEST ROC=0.7997 | ACC=0.8924
  seed=5 | TEST

### Real Septic Shock medical & competition "Give Me Some Credit" datasets:

Load Datasets:

In [ ]:
# Run, only once, for Septic Shock
from google.colab import files
uploaded = files.upload()

Saving septic for DL.xlsx to septic for DL.xlsx


In [ ]:
# Run, only once, for Give Me Some Credit
from google.colab import files
uploaded = files.upload()

Saving give me some credit dataset.csv to give me some credit dataset.csv


In [ ]:
# SEPTIC SHOCK

# 1. Load and preprocess data
septic = pd.read_excel("septic for DL.xlsx")
print(septic.shape)
display(septic.head())

# remove index column if it exists
if "Unnamed: 0" in septic.columns:
    septic = septic.drop(columns=["Unnamed: 0"])

SEPTIC_TARGET_COL = "died_in_1month_lbl"

# deduplicate
septic = deduplicate_dataframe(septic)

# categorical columns
SEPTIC_CAT_COLS = [
    "gender",
    "nationality",
    "cultures-result_m2_p3d",
]

# 2. Split data
sp_train_df, sp_val_df, sp_test_df = split_train_val_test(
    septic,
    SEPTIC_TARGET_COL,
    test_size=0.1,
    val_size=0.1,
    random_state=42,
    stratify=True
)

# 3. Impute missing values
SEPTIC_NUM_COLS = [
    c for c in sp_train_df.columns
    if c != SEPTIC_TARGET_COL and c not in SEPTIC_CAT_COLS
]

septic_imputed = impute_splits_with_verification(
    train_df=sp_train_df,
    val_df=sp_val_df,
    test_df=sp_test_df,
    target_col=SEPTIC_TARGET_COL,
    explicit_cat_cols=SEPTIC_CAT_COLS,
    explicit_num_cols=SEPTIC_NUM_COLS
)

sp_tr = septic_imputed["train"]
sp_va = septic_imputed["val"]
sp_te = septic_imputed["test"]

# 4. Encode categoricals
enc_septic = fit_categorical_encoders(sp_tr, SEPTIC_CAT_COLS)

Xtr_cat = transform_categoricals(sp_tr, enc_septic)
Xva_cat = transform_categoricals(sp_va, enc_septic)
Xte_cat = transform_categoricals(sp_te, enc_septic)

# 5. Extract numerics
Xtr_num = sp_tr[SEPTIC_NUM_COLS].values.astype(np.float32)
Xva_num = sp_va[SEPTIC_NUM_COLS].values.astype(np.float32)
Xte_num = sp_te[SEPTIC_NUM_COLS].values.astype(np.float32)

# 6. Extract targets
ytr = sp_tr[SEPTIC_TARGET_COL].values
yva = sp_va[SEPTIC_TARGET_COL].values
yte = sp_te[SEPTIC_TARGET_COL].values

# 7. Run experiment
SEPTIC_RESULTS = run_dataset_experiment(
    dataset_name="SepticShock",
    Xtr_cat=Xtr_cat, Xtr_num=Xtr_num, ytr=ytr,
    Xva_cat=Xva_cat, Xva_num=Xva_num, yva=yva,
    Xte_cat=Xte_cat, Xte_num=Xte_num, yte=yte,
    cardinalities=enc_septic["cardinalities"],
    train_df=sp_tr,
    val_df=sp_va,
    test_df=sp_te,
    target_col=SEPTIC_TARGET_COL,
    cat_cols=SEPTIC_CAT_COLS,
    seeds=[0,1,2,3,4,5,6,7,8,9],
    epochs=40
)

# 8. Access results
print("\n=== FINAL RESULTS ===")
for method, results in SEPTIC_RESULTS.items():
    if method != "_time_minutes":
        rocs = [r["test_roc_auc"] for r in results]
        print(f"{method}: {np.mean(rocs):.4f} ± {np.std(rocs, ddof=1):.4f}")

(5579, 29)


,age_at_t0,gender,nationality,hemoglobin-numeric result_1_1d,albumin-numeric result_1_1d,creat-numeric result_1_1d,crp -numeric result_1_1d,wbc -numeric result_1_1d,egfr-adm,cultures-result_m2_p3d,...,arb_1d,ca_1d,dm_1d,pvd_1d,chf_1d,insulin_1d,copd_1d,ckd_1d,mechanical_vent_1d,died_in_1month_lbl
0,82.57,Male,Unspecified,13.31,2.4,1.82,283.51,10.19,33.709037,none,...,0,0,1,0,0,0,0,0,0,1
1,89.70,Female,Unspecified,NaN,1.8,1.10,NaN,NaN,43.493977,none,...,0,0,0,0,0,0,0,1,1,1
2,83.52,Male,Jew,9.54,3.3,2.35,NaN,87.21,24.583579,none,...,0,1,0,0,0,0,1,1,0,0
3,73.98,Male,Jew,7.50,2.9,1.00,27.10,5.42,73.847861,none,...,0,0,0,1,0,0,0,0,0,1
4,87.92,Male,Unspecified,11.49,2.3,3.67,223.68,7.61,13.904825,none,...,1,1,0,0,0,1,0,0,0,1


Original shape: (5579, 29)
Duplicate rows: 0
Duplicate percentage: 0.00%
After deduplication: (5579, 29)

Running SepticShock - RAW...
  seed=0 | TEST ROC=0.6597 | ACC=0.6649
  seed=1 | TEST ROC=0.6660 | ACC=0.6559
  seed=2 | TEST ROC=0.6702 | ACC=0.6452
  seed=3 | TEST ROC=0.6458 | ACC=0.6308
  seed=4 | TEST ROC=0.6665 | ACC=0.6505
  seed=5 | TEST ROC=0.6824 | ACC=0.6595
  seed=6 | TEST ROC=0.6727 | ACC=0.6505
  seed=7 | TEST ROC=0.6702 | ACC=0.6523
  seed=8 | TEST ROC=0.6624 | ACC=0.6434
  seed=9 | TEST ROC=0.6731 | ACC=0.6434

=== SepticShock_RAW Summary (mean ± std over seeds) ===
TEST ROC-AUC: 0.6669 ± 0.0097
TEST ACC:     0.6496 ± 0.0096
Time: 12.27 min

Running SepticShock - ZSCORE...
  seed=0 | TEST ROC=0.6807 | ACC=0.6631
  seed=1 | TEST ROC=0.6959 | ACC=0.6792
  seed=2 | TEST ROC=0.7042 | ACC=0.6810
  seed=3 | TEST ROC=0.6965 | ACC=0.6864
  seed=4 | TEST ROC=0.6938 | ACC=0.6685
  seed=5 | TEST ROC=0.6938 | ACC=0.6505
  seed=6 | TEST ROC=0.6948 | ACC=0.6756
  seed=7 | TEST ROC

In [ ]:
# Safe version for Give Me Some Credit
# RF matrix builder - robust to zero categorical columns
# Safe version for datasets that may have zero categorical columns

def build_rf_matrices_safe(
    train_df,
    val_df,
    test_df,
    target_col,
    cat_cols
):
    cat_cols = list(cat_cols) if cat_cols is not None else []
    num_cols = [c for c in train_df.columns if c != target_col and c not in cat_cols]

    Xtr_parts = []
    Xva_parts = []
    Xte_parts = []

    # categorical part
    if len(cat_cols) > 0:
        tr_cat = pd.get_dummies(train_df[cat_cols], columns=cat_cols, dummy_na=False)
        va_cat = pd.get_dummies(val_df[cat_cols], columns=cat_cols, dummy_na=False)
        te_cat = pd.get_dummies(test_df[cat_cols], columns=cat_cols, dummy_na=False)

        all_cat_cols = sorted(set(tr_cat.columns) | set(va_cat.columns) | set(te_cat.columns))
        tr_cat = tr_cat.reindex(columns=all_cat_cols, fill_value=0)
        va_cat = va_cat.reindex(columns=all_cat_cols, fill_value=0)
        te_cat = te_cat.reindex(columns=all_cat_cols, fill_value=0)

        Xtr_parts.append(tr_cat.astype(np.float32))
        Xva_parts.append(va_cat.astype(np.float32))
        Xte_parts.append(te_cat.astype(np.float32))

    # numeric part
    if len(num_cols) > 0:
        Xtr_parts.append(train_df[num_cols].astype(np.float32))
        Xva_parts.append(val_df[num_cols].astype(np.float32))
        Xte_parts.append(test_df[num_cols].astype(np.float32))

    if len(Xtr_parts) == 0:
        raise ValueError("No feature columns found for RF.")

    Xtr = pd.concat(Xtr_parts, axis=1).to_numpy(dtype=np.float32)
    Xva = pd.concat(Xva_parts, axis=1).to_numpy(dtype=np.float32)
    Xte = pd.concat(Xte_parts, axis=1).to_numpy(dtype=np.float32)

    ytr = train_df[target_col].to_numpy()
    yva = val_df[target_col].to_numpy()
    yte = test_df[target_col].to_numpy()

    return {
        "Xtr": Xtr,
        "Xva": Xva,
        "Xte": Xte,
        "ytr": ytr,
        "yva": yva,
        "yte": yte,
        "n_features": Xtr.shape[1]
    }


def transform_categoricals_safe(df, encoders_dict):
    cardinalities = encoders_dict.get("cardinalities", [])
    n_rows = len(df)

    if len(cardinalities) == 0:
        return np.zeros((n_rows, 0), dtype=np.int64)

    return transform_categoricals(df, encoders_dict)


def run_dataset_experiment_safe(
    dataset_name,
    Xtr_cat, Xtr_num, ytr,
    Xva_cat, Xva_num, yva,
    Xte_cat, Xte_num, yte,
    cardinalities,
    *,
    train_df, val_df, test_df, target_col, cat_cols,
    ftt_dim=32, ftt_depth=3, ftt_heads=8, ftt_attn_dropout=0.2,
    ftt_ff_dropout=0.2, ple_n_bins=20,
    rf_n_estimators=100, rf_max_depth=None, rf_min_samples_leaf=1,
    rf_max_features="sqrt", rf_n_jobs=-1,
    seeds=(0, 1, 2), epochs=40
):
    """
    Runs complete experiment for one dataset:
    1. FT-Transformer with RAW numerics
    2. FT-Transformer with Z-score normalization
    3. FT-Transformer with Rank transformation
    4. FT-Transformer with PLE embeddings
    5. Random Forest baseline

    Safe for datasets with zero categorical columns.
    """
    all_results = {}
    all_times = {}

    cardinalities = list(cardinalities) if cardinalities is not None else []

    # standard FT-Transformer for RAW / Z / RANK
    def build_standard_ftt(num_continuous):
        return FTTransformer(
            categories=tuple(cardinalities),
            num_continuous=num_continuous,
            dim=ftt_dim,
            dim_out=1,
            depth=ftt_depth,
            heads=ftt_heads,
            attn_dropout=ftt_attn_dropout,
            ff_dropout=ftt_ff_dropout
        )


    # 1) RAW
    res, time_min = run_ftt_method(
        "RAW",
        dataset_name,
        cardinalities,
        Xtr_cat, Xtr_num, ytr,
        Xva_cat, Xva_num, yva,
        Xte_cat, Xte_num, yte,
        lambda: build_standard_ftt(Xtr_num.shape[1]),
        seeds,
        epochs
    )
    all_results["FTT_RAW"] = res
    all_times["FTT_RAW"] = time_min


    # 2) Z-SCORE
    z_scaler = fit_zscore_scaler(Xtr_num)
    Xtr_z = transform_zscore(Xtr_num, z_scaler)
    Xva_z = transform_zscore(Xva_num, z_scaler)
    Xte_z = transform_zscore(Xte_num, z_scaler)

    res, time_min = run_ftt_method(
        "ZSCORE",
        dataset_name,
        cardinalities,
        Xtr_cat, Xtr_z, ytr,
        Xva_cat, Xva_z, yva,
        Xte_cat, Xte_z, yte,
        lambda: build_standard_ftt(Xtr_z.shape[1]),
        seeds,
        epochs
    )
    all_results["FTT_ZSCORE"] = res
    all_times["FTT_ZSCORE"] = time_min


    # 3) RANK
    rank_scaler = fit_rank_transformer(Xtr_num)
    Xtr_rank = transform_to_rank(Xtr_num, rank_scaler)
    Xva_rank = transform_to_rank(Xva_num, rank_scaler)
    Xte_rank = transform_to_rank(Xte_num, rank_scaler)

    res, time_min = run_ftt_method(
        "RANK",
        dataset_name,
        cardinalities,
        Xtr_cat, Xtr_rank, ytr,
        Xva_cat, Xva_rank, yva,
        Xte_cat, Xte_rank, yte,
        lambda: build_standard_ftt(Xtr_rank.shape[1]),
        seeds,
        epochs
    )
    all_results["FTT_RANK"] = res
    all_times["FTT_RANK"] = time_min


    # 4) PLE
    # FTTransformer cannot receive zero total tokens.
    # If there are no real categorical features, inject one dummy categorical token.
    if len(cardinalities) == 0:
        ple_cardinalities = [1]
        Xtr_cat_ple = np.zeros((len(Xtr_num), 1), dtype=np.int64)
        Xva_cat_ple = np.zeros((len(Xva_num), 1), dtype=np.int64)
        Xte_cat_ple = np.zeros((len(Xte_num), 1), dtype=np.int64)
    else:
        ple_cardinalities = cardinalities
        Xtr_cat_ple = Xtr_cat
        Xva_cat_ple = Xva_cat
        Xte_cat_ple = Xte_cat

    def build_ple_ftt():
        edges = fit_ple_edges_quantiles(Xtr_num, n_bins=ple_n_bins)

        base = FTTransformer(
            categories=tuple(ple_cardinalities),
            num_continuous=0,
            dim=ftt_dim,
            dim_out=1,
            depth=ftt_depth,
            heads=ftt_heads,
            attn_dropout=ftt_attn_dropout,
            ff_dropout=ftt_ff_dropout,
        )

        ple_embedder = PLENumericTokenEmbedder(edges=edges, dim=ftt_dim)
        return FTTransformerWithPLE(base, ple_embedder)

    res, time_min = run_ftt_method(
        "PLE",
        dataset_name,
        ple_cardinalities,
        Xtr_cat_ple, Xtr_num, ytr,
        Xva_cat_ple, Xva_num, yva,
        Xte_cat_ple, Xte_num, yte,
        build_ple_ftt,
        seeds,
        epochs
    )
    all_results["FTT_PLE"] = res
    all_times["FTT_PLE"] = time_min


    # 5) Random Forest baseline
    if train_df is not None:
        print(f"\nRunning {dataset_name} - RF_RAW...")
        t0 = time.time()

        mats = build_rf_matrices_safe(
            train_df=train_df,
            val_df=val_df,
            test_df=test_df,
            target_col=target_col,
            cat_cols=cat_cols
        )

        print(f"  RF feature count: {mats['n_features']}")

        res_rf = []
        for s in seeds:
            r = run_one_seed_rf(
                seed=s,
                Xtr=mats["Xtr"], ytr=mats["ytr"],
                Xva=mats["Xva"], yva=mats["yva"],
                Xte=mats["Xte"], yte=mats["yte"],
                n_estimators=rf_n_estimators,
                max_depth=rf_max_depth,
                min_samples_leaf=rf_min_samples_leaf,
                max_features=rf_max_features,
                n_jobs=rf_n_jobs
            )

            res_rf.append(r)
            print(f"  seed={s} | TEST ROC={r['test_roc_auc']:.4f} | ACC={r['test_accuracy']:.4f}")

        elapsed_min = (time.time() - t0) / 60.0
        all_times["RF_RAW"] = elapsed_min

        summarize_results(f"{dataset_name}_RF_RAW", res_rf)
        print(f"Time: {elapsed_min:.2f} min")

        all_results["RF_RAW"] = res_rf

    # store timing info after all methods are done
    all_results["_time_minutes"] = all_times

    return all_results


# Give Me Some Credit
credit_risk = pd.read_csv("give me some credit dataset.csv")
print(credit_risk.shape)

# 1. Load and preprocess data
if "Unnamed: 0" in credit_risk.columns:
    credit_risk = credit_risk.drop(columns=["Unnamed: 0"])

credit_risk = deduplicate_dataframe(credit_risk)

CREDIT_TARGET_COL = "SeriousDlqin2yrs"
CREDIT_CAT_COLS = []

credit_risk = credit_risk.dropna(subset=[CREDIT_TARGET_COL])

# 2. Split data
cr_train_df, cr_val_df, cr_test_df = split_train_val_test(
    credit_risk,
    CREDIT_TARGET_COL,
    test_size=0.1,
    val_size=0.1,
    random_state=42,
    stratify=True
)

# 3. Impute missing values
CREDIT_NUM_COLS = [
    c for c in cr_train_df.columns
    if c != CREDIT_TARGET_COL and c not in CREDIT_CAT_COLS
]

credit_imputed = impute_splits_with_verification(
    train_df=cr_train_df,
    val_df=cr_val_df,
    test_df=cr_test_df,
    target_col=CREDIT_TARGET_COL,
    explicit_cat_cols=CREDIT_CAT_COLS,
    explicit_num_cols=CREDIT_NUM_COLS
)

cr_tr = credit_imputed["train"]
cr_va = credit_imputed["val"]
cr_te = credit_imputed["test"]

# 4. Encode categoricals
enc_credit = fit_categorical_encoders(cr_tr, CREDIT_CAT_COLS)

Xtr_cat = transform_categoricals_safe(cr_tr, enc_credit)
Xva_cat = transform_categoricals_safe(cr_va, enc_credit)
Xte_cat = transform_categoricals_safe(cr_te, enc_credit)

# 5. Extract numerics
Xtr_num = cr_tr[CREDIT_NUM_COLS].to_numpy(dtype=np.float32)
Xva_num = cr_va[CREDIT_NUM_COLS].to_numpy(dtype=np.float32)
Xte_num = cr_te[CREDIT_NUM_COLS].to_numpy(dtype=np.float32)

# 6. Extract targets
ytr = cr_tr[CREDIT_TARGET_COL].to_numpy()
yva = cr_va[CREDIT_TARGET_COL].to_numpy()
yte = cr_te[CREDIT_TARGET_COL].to_numpy()

# 7. Run experiment
CREDIT_RESULTS = run_dataset_experiment_safe(
    dataset_name="GiveMeSomeCredit",
    Xtr_cat=Xtr_cat, Xtr_num=Xtr_num, ytr=ytr,
    Xva_cat=Xva_cat, Xva_num=Xva_num, yva=yva,
    Xte_cat=Xte_cat, Xte_num=Xte_num, yte=yte,
    cardinalities=enc_credit.get("cardinalities", []),
    train_df=cr_tr,
    val_df=cr_va,
    test_df=cr_te,
    target_col=CREDIT_TARGET_COL,
    cat_cols=CREDIT_CAT_COLS,
    seeds=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
    epochs=40
)

# 8. Access results
print("\n=== FINAL RESULTS ===")
for method, results in CREDIT_RESULTS.items():
    if method != "_time_minutes":
        rocs = [r["test_roc_auc"] for r in results]
        print(f"{method}: {np.mean(rocs):.4f} ± {np.std(rocs, ddof=1):.4f}")

(148359, 12)
Original shape: (148359, 11)
Duplicate rows: 47666
Duplicate percentage: 32.13%
After deduplication: (100693, 11)

Running GiveMeSomeCredit - RAW...
  seed=0 | TEST ROC=0.8562 | ACC=0.9376
  seed=1 | TEST ROC=0.8541 | ACC=0.9368
  seed=2 | TEST ROC=0.8566 | ACC=0.9364
  seed=3 | TEST ROC=0.8558 | ACC=0.9372
  seed=4 | TEST ROC=0.8556 | ACC=0.9381
  seed=5 | TEST ROC=0.8563 | ACC=0.9360
  seed=6 | TEST ROC=0.8570 | ACC=0.9385
  seed=7 | TEST ROC=0.8574 | ACC=0.9379
  seed=8 | TEST ROC=0.8565 | ACC=0.9383
  seed=9 | TEST ROC=0.8573 | ACC=0.9368

=== GiveMeSomeCredit_RAW Summary (mean ± std over seeds) ===
TEST ROC-AUC: 0.8563 ± 0.0010
TEST ACC:     0.9374 ± 0.0008
Time: 223.33 min

Running GiveMeSomeCredit - ZSCORE...
  seed=0 | TEST ROC=0.8523 | ACC=0.9362
  seed=1 | TEST ROC=0.8509 | ACC=0.9364
  seed=2 | TEST ROC=0.8429 | ACC=0.9368
  seed=3 | TEST ROC=0.8493 | ACC=0.9366
  seed=4 | TEST ROC=0.8486 | ACC=0.9366
  seed=5 | TEST ROC=0.8492 | ACC=0.9368
  seed=6 | TEST ROC=0

# **For report - All Datasets description**

### Functions and data load

In [ ]:
# Helper function for summary plot
def add_dataset_summary(summary_rows, dataset_name, df, target_col, cat_cols):
    feature_cols = [col for col in df.columns if col != target_col]
    num_cols = [col for col in feature_cols if col not in cat_cols]

    summary_rows.append({
        "Dataset": dataset_name,
        "Observations": df.shape[0],
        "Total Features": len(feature_cols),
        "Categorical Features": len(cat_cols),
        "Numerical Features": len(num_cols),
    })


dataset_summary_rows = []



# HEART DISEASE (UCI)
kaggle_dataset("johnsmith88/heart-disease-dataset", f"{DATA_ROOT}/heart_disease")
!ls -lah {DATA_ROOT}/heart_disease | head

heart_disease = pd.read_csv(f"{DATA_ROOT}/heart_disease/heart.csv")
print(heart_disease.shape)
heart_disease.head()

heart_disease = deduplicate_dataframe(heart_disease)

HEART_DISEASE_TARGET_COL = "target"
HEART_DISEASE_CAT_COLS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]

add_dataset_summary(
    dataset_summary_rows,
    "Heart Disease (UCI)",
    heart_disease,
    HEART_DISEASE_TARGET_COL,
    HEART_DISEASE_CAT_COLS
)



# CREDIT CARD DEFAULT
kaggle_dataset(
    "uciml/default-of-credit-card-clients-dataset",
    f"{DATA_ROOT}/credit"
)
!ls -lah {DATA_ROOT}/credit | head

credit = pd.read_csv(f"{DATA_ROOT}/credit/UCI_Credit_Card.csv")
print(credit.shape)
credit.head()

CREDIT_TARGET_COL = "default.payment.next.month"
credit[CREDIT_TARGET_COL] = credit[CREDIT_TARGET_COL].astype(int)

CREDIT_CAT_COLS = [
    "SEX", "EDUCATION", "MARRIAGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"
]

# Drop ID (not a real feature)
credit = credit.drop(columns=["ID"])

# Remove duplicates
credit = deduplicate_dataframe(credit)

add_dataset_summary(
    dataset_summary_rows,
    "Credit Card Default",
    credit,
    CREDIT_TARGET_COL,
    CREDIT_CAT_COLS
)



# STROKE
kaggle_dataset("fedesoriano/stroke-prediction-dataset", f"{DATA_ROOT}/stroke")
!ls -lah {DATA_ROOT}/stroke | head

stroke = pd.read_csv(f"{DATA_ROOT}/stroke/healthcare-dataset-stroke-data.csv")
print(stroke.shape)
stroke.head()

stroke = deduplicate_dataframe(stroke)

STROKE_TARGET_COL = "stroke"
STROKE_CAT_COLS = [
    "gender",
    "ever_married",
    "work_type",
    "Residence_type",
    "smoking_status"
]

add_dataset_summary(
    dataset_summary_rows,
    "Stroke",
    stroke,
    STROKE_TARGET_COL,
    STROKE_CAT_COLS
)




# BANK MARKETING
kaggle_dataset(
    "mohitjanbandhu/bank-marketing-with-socialeconomic-context",
    f"{DATA_ROOT}/bank"
)
!ls -lah {DATA_ROOT}/bank | head

bank = pd.read_csv(f"{DATA_ROOT}/bank/bank-additional-full.csv", sep=";")
print(bank.shape)
bank.head()

bank = deduplicate_dataframe(bank)

# remove leakage feature
bank = bank.drop(columns=["duration"])

# convert target to 0/1
bank["y"] = (bank["y"] == "yes").astype(int)

BANK_TARGET_COL = "y"
BANK_CAT_COLS = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

add_dataset_summary(
    dataset_summary_rows,
    "Bank Marketing",
    bank,
    BANK_TARGET_COL,
    BANK_CAT_COLS
)

# GIVE ME SOME CREDIT

credit_risk = pd.read_csv("give me some credit dataset.csv", na_values=["NA"])
print(credit_risk.shape)
credit_risk.head()

# remove uploaded index column if it exists
if "Unnamed: 0" in credit_risk.columns:
    credit_risk = credit_risk.drop(columns=["Unnamed: 0"])

credit_risk = deduplicate_dataframe(credit_risk)

CREDIT_RISK_TARGET_COL = "SeriousDlqin2yrs"
CREDIT_RISK_CAT_COLS = []

add_dataset_summary(
    dataset_summary_rows,
    "Give Me Some Credit",
    credit_risk,
    CREDIT_RISK_TARGET_COL,
    CREDIT_RISK_CAT_COLS
)

# SEPTIC SHOCK

# 1. Load and preprocess data
septic = pd.read_excel("septic for DL.xlsx")
print(septic.shape)

# remove index column if it exists
if "Unnamed: 0" in septic.columns:
    septic = septic.drop(columns=["Unnamed: 0"])

SEPTIC_TARGET_COL = "died_in_1month_lbl"

# deduplicate
septic = deduplicate_dataframe(septic)
# categorical columns
SEPTIC_CAT_COLS = [
    "gender",
    "nationality",
    "cultures-result_m2_p3d",
]

add_dataset_summary(
    dataset_summary_rows,
    "Septic Shock (medical)",
    septic,
    SEPTIC_TARGET_COL,
    SEPTIC_CAT_COLS
)

Dataset URL: https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset
License(s): unknown
heart-disease-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
total 56K
drwxr-xr-x 2 root root 4.0K Mar  9 19:03 .
drwxr-xr-x 6 root root 4.0K Mar  9 19:03 ..
-rw-r--r-- 1 root root  38K Mar  9 19:04 heart.csv
-rw-r--r-- 1 root root 6.2K Oct 25  2019 heart-disease-dataset.zip
(1025, 14)
Original shape: (1025, 14)
Duplicate rows: 723
Duplicate percentage: 70.54%
After deduplication: (302, 14)
Dataset URL: https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
License(s): CC0-1.0
default-of-credit-card-clients-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
total 3.8M
drwxr-xr-x 2 root root  4.0K Mar  9 19:03 .
drwxr-xr-x 6 root root  4.0K Mar  9 19:03 ..
-rw-r--r-- 1 root root 1002K Sep 20  2019 default-of-credit-card-clients-dataset.zip
-rw-r--r-- 1 root root  2.8M Mar  9 19

,age_at_t0,gender,nationality,hemoglobin-numeric result_1_1d,albumin-numeric result_1_1d,creat-numeric result_1_1d,crp -numeric result_1_1d,wbc -numeric result_1_1d,egfr-adm,cultures-result_m2_p3d,...,arb_1d,ca_1d,dm_1d,pvd_1d,chf_1d,insulin_1d,copd_1d,ckd_1d,mechanical_vent_1d,died_in_1month_lbl
0,82.57,Male,Unspecified,13.31,2.4,1.82,283.51,10.19,33.709037,none,...,0,0,1,0,0,0,0,0,0,1
1,89.70,Female,Unspecified,NaN,1.8,1.10,NaN,NaN,43.493977,none,...,0,0,0,0,0,0,0,1,1,1
2,83.52,Male,Jew,9.54,3.3,2.35,NaN,87.21,24.583579,none,...,0,1,0,0,0,0,1,1,0,0
3,73.98,Male,Jew,7.50,2.9,1.00,27.10,5.42,73.847861,none,...,0,0,0,1,0,0,0,0,0,1
4,87.92,Male,Unspecified,11.49,2.3,3.67,223.68,7.61,13.904825,none,...,1,1,0,0,0,1,0,0,0,1


Original shape: (5579, 29)
Duplicate rows: 0
Duplicate percentage: 0.00%
After deduplication: (5579, 29)


### Summary table

In [ ]:
# FINAL SUMMARY TABLE
datasets_summary = pd.DataFrame(dataset_summary_rows)

datasets_summary = datasets_summary[
    ["Dataset", "Observations", "Total Features", "Categorical Features", "Numerical Features"]
]

datasets_summary = datasets_summary.sort_values(
    by="Observations",
    ascending=True
).reset_index(drop=True)

datasets_summary.index = datasets_summary.index + 1

display(datasets_summary)

,Dataset,Observations,Total Features,Categorical Features,Numerical Features
1,Heart Disease (UCI),302,13,8,5
2,Stroke,5110,11,5,6
3,Septic Shock (medical),5579,28,3,25
4,Credit Card Default,29965,23,9,14
5,Bank Marketing,41176,19,10,9
6,Give Me Some Credit,100693,10,0,10


# **Results Analysis - Friedman + Nemenyi**

### Functions setup

In [ ]:
# Friedman preprocess matrix

def build_friedman_matrix(all_results_by_dataset):
    rows = []

    for dataset_name, res in all_results_by_dataset.items():
        row = {"dataset": dataset_name}

        for method_name, runs in res.items():
            if method_name.startswith("_"): # to skip time significance evaluation
                continue

            if not isinstance(runs, list) or len(runs) == 0:
                raise ValueError(f"{dataset_name}:{method_name} is not a non-empty list (got {type(runs)})")

            if not isinstance(runs[0], dict) or "test_roc_auc" not in runs[0]:
                raise ValueError(
                    f"{dataset_name}:{method_name} entries must be dicts with 'test_roc_auc' "
                    f"(got {type(runs[0])}, keys={list(runs[0].keys()) if isinstance(runs[0], dict) else None})"
                )

            scores = [r["test_roc_auc"] for r in runs]
            row[method_name] = float(np.mean(scores))

        rows.append(row)

    return pd.DataFrame(rows).set_index("dataset")


In [ ]:
# Friedman test (omnibus)

# Input:
#   friedman_df : DataFrame
#       rows   = datasets
#       columns = methods (FTT_RAW, FTT_ZSCORE, FTT_RANK, FTT_PLE, RF_RAW)
#       values  = mean TEST ROC-AUC per dataset-method

# Output:
#   Friedman chi-square statistic and p-value

# Interpretation:
#   H0: All methods perform equally (no systematic difference)
#   If p < alpha (e.g. 0.05) → reject H0 and continue to Nemenyi

from scipy.stats import friedmanchisquare

def run_friedman_test(friedman_df):
    # Each column is one method evaluated across all datasets
    method_scores = [friedman_df[col].values for col in friedman_df.columns]

    stat, p_value = friedmanchisquare(*method_scores)

    print("=== Friedman test ===")
    print(f"Chi-square statistic: {stat:.4f}")
    print(f"p-value:              {p_value:.6f}")

    return stat, p_value

# Usage:
# stat, p = run_friedman_test(FRIEDMAN_DF)

In [ ]:
# Nemenyi post-hoc test
# We run it only if Friedman test showed us significance difference

# Input:
#   friedman_df : DataFrame (same as above)
#   alpha       : significance level (default 0.05)

# Output:
#   - Mean rank per method
#   - Critical Difference (CD)
#   - Pairwise significance table (True = significantly different)

# Notes:
#   - Lower rank = better performance
#   - Based on average ranks across datasets

import numpy as np
import pandas as pd
from scipy.stats import rankdata
from math import sqrt

def run_nemenyi_test(friedman_df, alpha=0.05):
    methods = friedman_df.columns.tolist()
    n_datasets = friedman_df.shape[0]
    k = len(methods)

    # Rank methods *within each dataset*
    ranks = friedman_df.rank(axis=1, ascending=False)

    # Mean rank per method
    mean_ranks = ranks.mean(axis=0)

    # Critical values q_alpha for Nemenyi (common choices)
    # These are standard for alpha=0.05
    q_alpha_table = {
        2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728,
        6: 2.850, 7: 2.949, 8: 3.031
    }

    if k not in q_alpha_table:
        raise ValueError("Nemenyi q_alpha not available for k > 8 methods.")

    q_alpha = q_alpha_table[k]

    # Critical Difference
    CD = q_alpha * sqrt(k * (k + 1) / (6 * n_datasets))

    print("=== Nemenyi post-hoc ===")
    print("Mean ranks (lower = better):")
    print(mean_ranks.sort_values())
    print(f"\nCritical Difference (CD): {CD:.4f}\n")

    # Pairwise comparison
    sig_matrix = pd.DataFrame(
        False, index=methods, columns=methods, dtype=bool
    )

    for i in range(k):
        for j in range(i + 1, k):
            diff = abs(mean_ranks.iloc[i] - mean_ranks.iloc[j])
            if diff > CD:
                sig_matrix.iloc[i, j] = True
                sig_matrix.iloc[j, i] = True

    return mean_ranks, CD, sig_matrix



### Run Friedman+Nemenyi significance tests

In [ ]:
# Results dict
ALL_RESULTS = {
    "Heart Disease": HEART_RESULTS,
    "Stroke": STROKE_RESULTS,
    "Septic Shock (medical)": SEPTIC_RESULTS,
    "Credit Card": CREDIT_RESULTS,
    "Bank Marketing": BANK_RESULTS,
    "Give Me Some Credit": CREDIT_RISK_RESULTS,
}

# Friedman input matrix
FRIEDMAN_DF = build_friedman_matrix(ALL_RESULTS)
print(FRIEDMAN_DF)


          FTT_RAW  FTT_ZSCORE  FTT_RANK   FTT_PLE    RF_RAW
dataset                                                    
Heart    0.875180    0.878427  0.881313  0.863276  0.882937
Adult    0.894960    0.900942  0.883815  0.902963  0.891228
Credit   0.757975    0.760246  0.760792  0.768891  0.759306
Bank     0.944256    0.942950  0.940235  0.947469  0.944400
=== Friedman test ===
Chi-square statistic: 3.0000
p-value:              0.557825


In [ ]:
# Insert the results per dataframe
# We do it manually because of runtime limitations

MANUAL_RESULTS = {
    "Heart Disease": {
        "FTT_RAW":    0.8538,
        "FTT_ZSCORE": 0.8576,
        "FTT_RANK":   0.8437,
        "FTT_PLE":    0.8529,
        "RF_RAW":     0.8370,
    },
    "Stroke": {
        "FTT_RAW":    0.6797,
        "FTT_ZSCORE": 0.7041,
        "FTT_RANK":   0.5459,
        "FTT_PLE":    0.7056,
        "RF_RAW":     0.7259,
    },
    "Septic Shock (medical)": {
        "FTT_RAW":    0.6669,
        "FTT_ZSCORE": 0.6949,
        "FTT_RANK":   0.6876,
        "FTT_PLE":    0.7207,
        "RF_RAW":     0.7232,
    },
    "Credit Card": {
        "FTT_RAW":    0.7496,
        "FTT_ZSCORE": 0.7576,
        "FTT_RANK":   0.7577,
        "FTT_PLE":    0.7671,
        "RF_RAW":     0.7542,
    },
    "Bank Marketing": {
        "FTT_RAW":    0.8013,
        "FTT_ZSCORE": 0.8006,
        "FTT_RANK":   0.8005,
        "FTT_PLE":    0.8022,
        "RF_RAW":     0.7718,
    },
    "Give Me Some Credit": {
        "FTT_RAW":    0.8563,
        "FTT_ZSCORE": 0.8494,
        "FTT_RANK":   0.7905,
        "FTT_PLE":    0.8540,
        "RF_RAW":     0.8243,
    }
}

# Build DataFrame from manual results
FRIEDMAN_DF = pd.DataFrame.from_dict(MANUAL_RESULTS, orient="index")

# Ensure column order matches previous Friedman runs
FRIEDMAN_DF = FRIEDMAN_DF[
    ["FTT_RAW", "FTT_ZSCORE", "FTT_RANK", "FTT_PLE", "RF_RAW"]
]


print("\n=== Friedman table ===")
print(FRIEDMAN_DF)



=== Friedman table ===
                        FTT_RAW  FTT_ZSCORE  FTT_RANK  FTT_PLE  RF_RAW
Heart Disease            0.8538      0.8576    0.8437   0.8529  0.8370
Stroke                   0.6797      0.7041    0.5459   0.7056  0.7259
Septic Shock (medical)   0.6669      0.6949    0.6876   0.7207  0.7232
Credit Card              0.7496      0.7576    0.7577   0.7671  0.7542
Bank Marketing           0.8013      0.8006    0.8005   0.8022  0.7718
Give Me Some Credit      0.8563      0.8494    0.7905   0.8540  0.8243


In [ ]:
# Run Friedman test - "Is there any statistically significant difference between methods?"
stat, p = run_friedman_test(FRIEDMAN_DF)

=== Friedman test ===
Chi-square statistic: 6.2667
p-value:              0.180099


In [ ]:
mean_ranks = FRIEDMAN_DF.rank(axis=1, ascending=False).mean().sort_values()
print(mean_ranks)


FTT_PLE       1.833333
FTT_ZSCORE    2.666667
FTT_RAW       3.166667
RF_RAW        3.333333
FTT_RANK      4.000000
dtype: float64


In [ ]:
# Nemenyi post-hoc
# Only run if Friedman is significant.

mean_ranks, CD, sig_pairs = run_nemenyi_test(FRIEDMAN_DF)

mean_ranks.sort_values()
+

=== Nemenyi post-hoc ===
Mean ranks (lower = better):
FTT_PLE       1.833333
FTT_ZSCORE    2.666667
FTT_RAW       3.166667
RF_RAW        3.333333
FTT_RANK      4.000000
dtype: float64

Critical Difference (CD): 2.4903



,0
FTT_PLE,1.833333
FTT_ZSCORE,2.666667
FTT_RAW,3.166667
RF_RAW,3.333333
FTT_RANK,4.000000
